In [1]:
import glob
import json
import os
import re
import sys

import nest_asyncio
from dotenv import load_dotenv

from skdecide import rollout
from skdecide.hub.solver.openevolve import (
    IntegratedOpenEvolve,
)

# patch asyncio so that applications using async functions can run in jupyter
nest_asyncio.apply()

load_dotenv()

beluga_toolkit_repo = os.path.abspath(os.environ["BELUGA_TOOLKIT_REPO"])
sys.path.append(beluga_toolkit_repo)

from beluga_lib.beluga_problem import BelugaProblemDecoder
from skd_domains.skd_pddl_domain import SkdPDDLDomain

In [2]:
# PARAMS
MAX_STEPS_BY_JIG = 10  # rollout max step (to be mult by n_jigs)
N_PB_EVAL = 1  # nb of pb to use for evaluation
N_PB_VALID = 3  # nb of pbs to use for final validation
PICK_PB_RANDOM = False  # pick random pbs or sequentially


# "install" beluga toolkit
beluga_toolkit_repo = os.path.abspath(os.environ["BELUGA_TOOLKIT_REPO"])
sys.path.append(beluga_toolkit_repo)
from beluga_lib.beluga_problem import BelugaProblemDecoder
from skd_domains.skd_pddl_domain import SkdPDDLDomain


# list json def files
def extract_problem_id(problem_file: str) -> int:
    problem_name = os.path.basename(problem_file)
    return int(re.match("problem_([0-9]+)", problem_name).group(1))


beluga_benchmark_dir = os.environ["BELUGA_BENCHMARK_DIR"]
beluga_json_files = sorted(
    glob.glob(f"{beluga_benchmark_dir}/*.json"), key=extract_problem_id
)

beluga_json_main = beluga_json_files[0]
if PICK_PB_RANDOM:
    raise NotImplementedError()
else:
    beluga_jsons_eval = beluga_json_files[:N_PB_EVAL]
    beluga_jsons_valid = beluga_json_files[N_PB_EVAL : N_PB_EVAL + N_PB_VALID]


def create_domain(problem_json: str, classic: bool = False) -> SkdPDDLDomain:
    problem_folder = os.path.dirname(problem_json)
    problem_name = os.path.basename(problem_json)
    with open(problem_json, "r") as fp:
        inst = json.load(fp, cls=BelugaProblemDecoder)
    domain = SkdPDDLDomain(inst, problem_name, problem_folder, classic=classic)
    domain.n_jigs = len(inst.jigs)
    return domain


# config, output, save path
example_dir = os.getcwd()
config = f"{example_dir}/config.yaml"
output_dir = f"{example_dir}/output/demo"

save_dir = f"{output_dir}/final"


domain_factory = lambda: create_domain(beluga_json_main)
domain = domain_factory()  # generate the pddl files

# domain factories on which evaluate
evaluator_domains = [create_domain(problem_json) for problem_json in beluga_jsons_eval]

# retrieve domain pddl def to inject in prompt  (created when instanciating the skdecide domains)
with open(f"{os.path.dirname(beluga_json_main)}/domain.pddl", "rt") as f:
    pddl_domain_def = f.read()


def prompt_update_function(system_message: str) -> str:
    return system_message.replace(
        "[INSERT_RAW_PDDL_DOMAIN_FILE_CONTENT_HERE]", pddl_domain_def
    )


def evaluator_rollout_max_steps(domain):
    return domain.n_jigs * MAX_STEPS_BY_JIG


initial_program_path = f"{example_dir}/advanced_program.py"


solver_kwargs = dict(
    domain_factory=domain_factory,
    evaluator_domains=evaluator_domains,
    evaluator_rollout_max_steps=evaluator_rollout_max_steps,
    config=config,
    output_dir=output_dir,
    evaluator_timeout=120,
    evaluator_rollout_num_episodes=1,
    prompt_add_public_api_instruction=False,
    prompt_include_public_api=False,
    prompt_add_blockevolve_instruction=False,
    prompt_update_function=prompt_update_function,
)

In [3]:
solver = IntegratedOpenEvolve(**solver_kwargs)
total_cost = 0
rollout_domain = domain_factory()
max_steps = evaluator_rollout_max_steps(rollout_domain)
render = False
# Instantiate a new solver with the proper domain factory so that
# the wrapped evolved planner works with the characteristics of the new domain
episodes = rollout(
    solver=solver,
    domain=rollout_domain,
    render=render,
    verbose=False,
    num_episodes=3,
    max_steps=max_steps,
    return_episodes=True,
)
total_cost += (
    sum(sum(c.cost for c in ep_costs) for ep_obs, ep_actions, ep_costs in episodes)
    / max_steps
)  # normalized by maze size
print(total_cost)

2026-03-04 16:07:26,153 - INFO - Logging to /home/nolwen/Projects/scikit-decide/examples/openevolve/beluga2/output/demo/logs/openevolve_20260304_160726.log
2026-03-04 16:07:26,155 - INFO - Set random seed to 42 for reproducibility
2026-03-04 16:07:26,198 - INFO - Initialized OpenAI LLM with model: gpt-oss-120b
2026-03-04 16:07:26,210 - INFO - Initialized OpenAI LLM with model: llama3.1-8b
2026-03-04 16:07:26,211 - INFO - Initialized LLM ensemble with models: gpt-oss-120b (weight: 0.50), llama3.1-8b (weight: 0.50)
2026-03-04 16:07:26,236 - INFO - Initialized LLM ensemble with models: gpt-oss-120b (weight: 0.50), llama3.1-8b (weight: 0.50)
2026-03-04 16:07:26,239 - INFO - Initialized prompt sampler
2026-03-04 16:07:26,242 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-03-04 16:07:26,243 - INFO - Initialized program database with 0 programs
2026-03-04 16:07:26,246 - INFO - Successfully loaded evaluation function from /home/nolwen/Projects/scikit-decide/exam

3.0


In [7]:
for episode in episodes:
    last_obs = episode[0][-1]
    print(domain.is_goal(last_obs))

False
False
False


In [3]:
beluga_benchmark_dir = os.environ["BELUGA_BENCHMARK_DIR"]
beluga_json_files = sorted(
    glob.glob(f"{beluga_benchmark_dir}/*.json"), key=extract_problem_id
)

n_pb_eval = 3
beluga_json_main = beluga_json_files[0]
beluga_json_eval = beluga_json_files[:n_pb_eval]

In [36]:
problem_json = beluga_json_main


def create_domain(problem_json: str, classic: bool = True) -> SkdPDDLDomain:
    problem_folder = os.path.dirname(problem_json)
    problem_name = os.path.basename(problem_json)
    with open(problem_json, "r") as fp:
        inst = json.load(fp, cls=BelugaProblemDecoder)
    domain = SkdPDDLDomain(inst, problem_name, problem_folder, classic=classic)
    domain.n_jigs = len(inst.jigs)
    return domain

In [49]:
d = create_domain(problem_json, False)

In [11]:
results = [(5.0, 1), (1.2, 0), (6.0, 2)]
results_by_type = list(zip(*results))
[sum(results_type) for results_type in results_by_type]

[12.2, 3]

In [10]:
from IPython.display import Markdown, display

print(
    (
        "You are an expert mathematician specialized in planning and scheduling.\nYour task is to improve a function that suggests the next action to take given the current position in a pddl domain.\nIt will be applied to a real-world logistic planning problem dealing with the storage and management of cargo transported by Airbus Beluga aircrafts.\n\nAircraft parts are held on *jigs*, which can slide and be stored on the *racks*.\n*Trailers* are used as special moving racks to transfer the jigs between the Beluga\nand the fixed racks. When the aircraft parts are sent to the production lines,\nthey transit to hangars where cranes remove the parts from the jigs.\nThe parts are then sent to production and the jigs return empty to the rack system.\n\nThe racks can contain several jigs in sequence, but only the jigs which are at the edges of the racks\n(either factory side or Beluga side) can be pulled out from the racks. This might require swapping jigs\nlocated at the rack edges to other racks in order to free the pass to jigs which are strictly inside the racks\n(i.e. not at their edges).\n\nConcerning the empty jigs to be returned to the Beluga, we only need to know their types\n(a jig type being a class of jigs of same loaded size and same empty size) and how many such\nempty jig types must be returned.\n\nWhen the Beluga lands on the production site, 2 high-level tasks must be performed:\n- unloading the parts (held on jigs) from the Beluga and storing them in the rack system ;\n- unstoring empty jigs from the rack system and loading them into the Beluga.\n\nBetween 2 Beluga flights, 3 high-level tasks must be considered:\n- unstoring parts held on jigs from the rack system and sending them to the production lines via the craning hangars ;\n- sending back empty jigs from the craning hangars to the rack system;\n- [optionally] swapping the jigs which are at the edges of the racks (either factory side or Beluga side) from one rack to another, in order to free the pass to jigs which are strictly inside the racks\n\nYour sampling function will be evaluated by applying it on several instances of the domain and computing the resulting average cost.\nMore precisely, given a maze, a new action is sampled and applied to the problem until the goal is reached\nor until a max number of steps related to the domain size (i.e. the number of jigs) is overcome. We take the average of the total cost on each test problem.\n\nThe heuristic is wrapped into a planner that can be used to stores information about the current problem from previous calls\nto increase the computation speed.\n\nYou must only perform edits between the # EVOLVE-BLOCK-START and # EVOLVE-BLOCK-END comments.\n\nYou are not allowed to use private attributes or methods of the domain, only its public API.\n\nFor context, here is the public api of <class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'> and of other relevant classes:\n## API Reference: `<class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'>`\n\n### Domain Capabilities (High-Level)\n- **Agent**: SingleAgent -> it is single-agent (i.e hosting only one agent).\n- **Concurrency**: Sequential -> its events/actions are sequential (non-parallel).\n- **Dynamics**: DeterministicTransitions -> its dynamics is deterministic and provided as a white-box model.\n- **Events**: Actions -> it handles only actions (i.e. controllable events).\n- **Goals**: Goals -> it has formalized goals.\n- **Initialization**: DeterministicInitialized -> it has a deterministic initial state known as white-box.\n- **Memory**: Markovian -> only its last state must be stored to compute its dynamics (pure Markovian\n- **Observability**: FullyObservable -> it is fully observable.\n- **Value**: PositiveCosts -> it sends only positive costs (i.e. negative rewards).\n\n### Domain base types\n- **T_event**: <class 'skd_domains.skd_base_domain.Action'>\n- **T_info**: None\n- **T_observation**: <class 'skd_domains.skd_base_domain.State'>\n- **T_predicate**: <class 'bool'>\n- **T_state**: <class 'skd_domains.skd_base_domain.State'>\n- **T_value**: <class 'float'>\n\n### Action space per agent\n<class 'skd_domains.skd_base_domain.ActionSpace'>\n\n### Observation space per agent\n<class 'skd_domains.skd_base_domain.ObservationSpace'>\n\n### Description\nDeterministic planning scikit-decide domain, based on a encoding of the actions to model the logics of\nthe transition function. The methods of this class should not be used directly, but rather through the\npublic methods of the domain feature class from which this class derives:\nhttps://airbus.github.io/scikit-decide/reference/_skdecide.domains.htmldeterministicplanningdomain\n\nArgs:\n    SkdBaseDomain (_type_): Base domain\n    DeterministicPlanningDomain (_type_): Compound scikit-decide class importing deterministic planning domain features\n\n### Attributes\n- **domain_str** (typing.Any): \n- **cost_functions** (set[int]): \n- **domain_path** (typing.Any): \n- **temp_pddl_directory** (<class 'NoneType'>): \n- **action_space** (typing.Any): \n- **transition_cost** (dict[tuple[State, Action, State], int]): \n- **observation_space** (typing.Any): \n- **task** (Task): \n- **problem_path** (typing.Any): \n- **succ_gen** (SuccessorGenerator): \n- **aops_gen** (ApplicableActionsGenerator): \n- **total_cost** (int | None): \n- **check_goal** (GoalChecker): \n\n### Methods\n#### - `__init__(self, beluga_problem: beluga_lib.beluga_problem.BelugaProblem, problem_name: str, instance_dir: os.PathLike = None, classic: bool = True, initial_state: beluga_lib.problem_state.BelugaProblemState = None) -> None`\n\n#### - `check_value(self, value: 'Value[D.T_value]') -> 'bool'`\nCheck that a value is compliant with its reward specification.\n\n**TIP:** This function returns always True by default because any kind of reward should be accepted at this level.\n\n##### Parameters\n- **value**: The value to check.\n\n##### Returns\nTrue if the value is compliant (False otherwise).\n\n#### - `cleanup(self)`\nErases the temporary directory containing PDDL files (if any)\n\n#### - `deserialize_action(self, action: tuple[str]) -> skd_domains.skd_base_domain.Action`\n\n#### - `deserialize_state(self, atoms: Iterable[tuple[str]], fluents: Iterable[tuple[tuple[str], int]])`\n\n#### - `get_action_mask(self, memory: 'Optional[D.T_state]' = None) -> 'Mask'`\nGet action mask for the given memory or internal one if omitted.\n\nAn action mask is another (more specific) format for applicable actions, that has a meaning only if the action\nspace can be iterated over in some way. It is represented by a flat array of 0's and 1's ordered as the actions\nwhen enumerated: 1 for an applicable action, and 0 for a not applicable action.\n\nThe action mask is used for instance by RL solvers to shut down logits associated to non-applicable actions in\nthe output of their internal neural network.\n\n##### Parameters\n- **memory**: The memory to consider. If None, works on the internal memory of the domain.\n\n##### Returns\na numpy array (or dict agent-> numpy array for multi-agent domains) with 0-1 indicating applicability of\nthe action (1 meaning applicable and 0 not applicable)\n\n#### - `get_action_space(self) -> 'Space[D.T_event]'`\nGet the (cached) domain action space (finite or infinite set).\n\n##### Returns\nThe action space.\n\n#### - `get_agents(self) -> 'set[str]'`\nReturn a singleton for single agent domains.\n\nWe must be here consistent with `skdecide.core.autocast()` which transforms a single agent domain\ninto a multi agents domain whose only agent has the id \"agent\".\n\n#### - `get_applicable_actions(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`\nGet the space (finite or infinite set) of applicable actions in the given memory (state or history), or in\nthe internal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nThe space of applicable actions.\n\n#### - `get_enabled_events(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`\nGet the space (finite or infinite set) of enabled uncontrollable events in the given memory (state or\nhistory), or in the internal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nThe space of enabled events.\n\n#### - `get_goals(self) -> 'Space[D.T_observation]'`\nGet the (cached) domain goals space (finite or infinite set).\n\n**WARNING:** Goal states are assumed to be fully observable (i.e. observation = state) so that there is never uncertainty\nabout whether the goal has been reached or not. This assumption guarantees that any policy that does not\nreach the goal with certainty incurs in infinite expected cost. - *Geffner, 2013: A Concise Introduction to\nModels and Methods for Automated Planning*\n\n##### Returns\nThe goals space.\n\n#### - `get_initial_state(self) -> 'D.T_state'`\nGet the (cached) initial state.\n\n##### Returns\nThe initial state.\n\n#### - `get_initial_state_distribution(self) -> 'Distribution[D.T_state]'`\nGet the (cached) probability distribution of initial states.\n\n##### Returns\nThe probability distribution of initial states.\n\n#### - `get_next_state(self, memory: 'D.T_state', action: 'D.T_event') -> 'D.T_state'`\nGet the next state given a memory and action.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe deterministic next state.\n\n#### - `get_next_state_distribution(self, memory: 'D.T_state', action: 'D.T_event') -> 'DiscreteDistribution[D.T_state]'`\nGet the discrete probability distribution of next state given a memory and action.\n\n**TIP:** In the Markovian case (memory only holds last state $s$), given an action $a$, this function can\nbe mathematically represented by $P(S'|s, a)$, where $S'$ is the next state random variable.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe discrete probability distribution of next state.\n\n#### - `get_observation(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'D.T_observation'`\nGet the deterministic observation given a state and action.\n\n##### Parameters\n- **state**: The state to be observed.\n- **action**: The last applied action (or None if the state is an initial state).\n\n##### Returns\nThe probability distribution of the observation.\n\n#### - `get_observation_distribution(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'Distribution[D.T_observation]'`\nGet the probability distribution of the observation given a state and action.\n\nIn mathematical terms (discrete case), given an action $a$, this function represents: $P(O|s, a)$,\nwhere $O$ is the random variable of the observation.\n\n##### Parameters\n- **state**: The state to be observed.\n- **action**: The last applied action (or None if the state is an initial state).\n\n##### Returns\nThe probability distribution of the observation.\n\n#### - `get_observation_space(self) -> 'Space[D.T_observation]'`\nGet the (cached) observation space (finite or infinite set).\n\n##### Returns\nThe observation space.\n\n#### - `get_pddl_domain(self) -> os.PathLike`\nGet the path to the PDDL domain file\n\nReturns:\n    os.PathLike: Path to the PDDL domain file\n\n#### - `get_pddl_problem(self) -> os.PathLike`\nGet the path to the PDDL problem file\n\nReturns:\n    os.PathLike: Path to the PDDL problem file\n\n#### - `get_transition_value(self, memory: 'D.T_state', action: 'D.T_event', next_state: 'Optional[D.T_state]' = None) -> 'Value[D.T_value]'`\nGet the value (reward or cost) of a transition.\n\nThe transition to consider is defined by the function parameters.\n\n**TIP:** If this function never depends on the next_state parameter for its computation, it is recommended to\nindicate it by overriding UncertainTransitions._is_transition_value_dependent_on_next_state_() to return\nFalse. This information can then be exploited by solvers to avoid computing next state to evaluate a\ntransition value (more efficient).\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n- **next_state**: The next state in which the transition ends (if needed for the computation).\n\n##### Returns\nThe transition value (reward or cost).\n\n#### - `is_action(self, event: 'D.T_event') -> 'bool'`\nIndicate whether an event is an action (i.e. a controllable event for the agents).\n\n**TIP:** \n\n##### Parameters\n- **event**: The event to consider.\n\n##### Returns\nTrue if the event is an action (False otherwise).\n\n#### - `is_applicable_action(self, action: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`\nIndicate whether an action is applicable in the given memory (state or history), or in the internal one if\nomitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nTrue if the action is applicable (False otherwise).\n\n#### - `is_enabled_event(self, event: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`\nIndicate whether an uncontrollable event is enabled in the given memory (state or history), or in the\ninternal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nTrue if the event is enabled (False otherwise).\n\n#### - `is_goal(self, observation: 'D.T_observation') -> 'D.T_predicate'`\nIndicate whether an observation belongs to the goals.\n\n**TIP:** \n\n##### Parameters\n- **observation**: The observation to consider.\n\n##### Returns\nTrue if the observation is a goal (False otherwise).\n\n#### - `is_observation(self, observation: 'D.T_observation') -> 'bool'`\nCheck that an observation indeed belongs to the domain observation space.\n\n**TIP:** \n\n##### Parameters\n- **observation**: The observation to consider.\n\n##### Returns\nTrue if the observation belongs to the domain observation space (False otherwise).\n\n#### - `is_terminal(self, state: 'D.T_state') -> 'D.T_predicate'`\nIndicate whether a state is terminal.\n\nA terminal state is a state with no outgoing transition (except to itself with value 0).\n\n##### Parameters\n- **state**: The state to consider.\n\n##### Returns\nTrue if the state is terminal (False otherwise).\n\n#### - `is_transition_value_dependent_on_next_state(self) -> 'bool'`\nIndicate whether get_transition_value() requires the next_state parameter for its computation (cached).\n\n##### Returns\nTrue if the transition value computation depends on next_state (False otherwise).\n\n#### - `reset(self) -> 'D.T_observation'`\nReset the state of the environment and return an initial observation.\n\n##### Returns\nAn initial observation.\n\n#### - `sample(self, memory: 'D.T_state', action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`\nSample one transition of the simulator's dynamics.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe environment outcome of the sampled transition.\n\n#### - `serialize_action(self, action: skd_domains.skd_base_domain.Action) -> tuple[str]`\n\n#### - `serialize_state(self, state: skd_domains.skd_base_domain.State) -> tuple[tuple[str], tuple[tuple[str], int]]`\n\n#### - `set_memory(self, memory: 'D.T_state') -> 'None'`\nSet internal memory attribute _memory to given one.\n\nThis can be useful to set a specific \"starting point\" before doing a rollout with successive Environment.step()\ncalls.\n\n##### Parameters\n- **memory**: The memory to set internally.\n\n##### Example\n```python\n# Set simulation_domain memory to my_state (assuming Markovian domain)\nsimulation_domain.set_memory(my_state)\n\n# Start a 100-steps rollout from here (applying my_action at every step)\nfor _ in range(100):\n    simulation_domain.step(my_action)\n```\n\n#### - `step(self, action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`\nRun one step of the environment's dynamics.\n\n**WARNING:** Before calling Environment.step() the first time or when the end of an episode is\nreached, Initializable.reset() must be called to reset the environment's state.\n\n##### Parameters\n- **action**: The action taken in the current memory (state or history) triggering the transition.\n\n##### Returns\nThe environment outcome of this step.\n\n## API Reference: `<class 'skdecide.core.EnvironmentOutcome'>`\n\n### Description\nAn environment outcome for an internal transition.\n\n#### Parameters\n- **observation**: The agent's observation of the current environment.\n- **value**: The value (reward or cost) returned after previous action.\n- **termination**: Whether the episode has ended, in which case further step() calls will return undefined results.\n- **info**: Optional auxiliary diagnostic information (helpful for debugging, and sometimes learning).\n\n### Attributes\n- **info** (Optional[D.T_info]): \n- **observation** (D.T_observation): \n- **value** (Optional[D.T_value]): \n- **termination** (Optional[D.T_predicate]): \n\n### Methods\n#### - `__init__(self, observation: 'D.T_observation', value: 'Optional[D.T_value]' = None, termination: 'Optional[D.T_predicate]' = None, info: 'Optional[D.T_info]' = None) -> None`\n\n#### - `asdict(self)`\nReturn the fields of the instance as a new dictionary mapping field names to field values.\n\n#### - `astuple(self)`\nReturn the fields of the instance as a new tuple of field values.\n\n#### - `replace(self, **changes)`\nReturn a new object replacing specified fields with new values.\n\n## API Reference: `<class 'skdecide.core.DiscreteDistribution'>`\n\n### Description\nA discrete probability distribution.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, values: 'list[tuple[T, float]]') -> 'None'`\nInitialize DiscreteDistribution.\n\n**TIP:** If the given probabilities do not sum to 1, they are implicitly normalized as such for sampling.\n\n##### Parameters\n- **values**: The list of (element, probability) pairs.\n\n##### Example\n```python\ngame_strategy = DiscreteDistribution([('rock', 0.7), ('paper', 0.1), ('scissors', 0.2)])\nmove = game_strategy.sample()\n```\n\n#### - `get_values(self) -> 'list[tuple[T, float]]'`\nGet the list of (element, probability) pairs.\n\n##### Returns\nThe (element, probability) pairs.\n\n#### - `sample(self) -> 'T'`\n\n## API Reference: `<class 'skdecide.core.Value'>`\n\n### Description\nA value (reward or cost).\n\n**WARNING:** It is recommended to use either the reward or the cost parameter. If no one is used, a reward/cost of 0 is\nassumed. If both are used, reward will be considered and cost ignored. In any case, both reward and cost\nattributes will be defined after initialization.\n\n#### Parameters\n- **reward**: The optional reward.\n- **cost**: The optional cost.\n\n#### Example\n```python\n# These two lines are equivalent, use the one you prefer\nvalue_1 = Value(reward=-5)\nvalue_2 = Value(cost=5)\n\nassert value_1.reward == value_2.reward == -5  # True\nassert value_1.cost == value_2.cost == 5  # True\n```\n\n### Attributes\n- **reward** (Optional[D.T_value]): \n- **cost** (Optional[D.T_value]): \n\n### Methods\n#### - `__init__(self, reward: 'Optional[D.T_value]' = None, cost: 'Optional[D.T_value]' = None) -> None`\n\n## API Reference: `<class 'skdecide.core.Distribution'>`\n\n### Description\nA probability distribution.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, /, *args, **kwargs)`\nInitialize self.  See help(type(self)) for accurate signature.\n\n#### - `sample(self) -> 'T'`\nSample from this distribution.\n\n##### Returns\nThe sampled element.\n\n## API Reference: `<class 'skdecide.core.Space'>`\n\n### Description\nA space representing a finite or infinite set.\n\nThis class (or any of its descendant) is typically used to specify action, observation or goal spaces.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, /, *args, **kwargs)`\nInitialize self.  See help(type(self)) for accurate signature.\n\n#### - `contains(self, x: 'T') -> 'bool'`\nCheck whether x is a valid member of this space.\n\n##### Parameters\n- **x**: The member to consider.\n\n##### Returns\nTrue if x is a valid member of this space (False otherwise)."
    )
)

You are an expert mathematician specialized in planning and scheduling.
Your task is to improve a function that suggests the next action to take given the current position in a pddl domain.
It will be applied to a real-world logistic planning problem dealing with the storage and management of cargo transported by Airbus Beluga aircrafts.

Aircraft parts are held on *jigs*, which can slide and be stored on the *racks*.
*Trailers* are used as special moving racks to transfer the jigs between the Beluga
and the fixed racks. When the aircraft parts are sent to the production lines,
they transit to hangars where cranes remove the parts from the jigs.
The parts are then sent to production and the jigs return empty to the rack system.

The racks can contain several jigs in sequence, but only the jigs which are at the edges of the racks
(either factory side or Beluga side) can be pulled out from the racks. This might require swapping jigs
located at the rack edges to other racks in order to fr

**Role:** You are an expert mathematician and PDDL planning specialist. 

**Task:** Improve the `sample_action` method in the `Planner` class to minimize the average cost of logistic operations for Airbus Beluga cargo.

# Domain Context: The Beluga Challenge
* **Jigs & Racks**: Aircraft parts are on jigs. Jigs slide into racks. Only jigs at the edges (factory or beluga side) can be moved.
* **Unblocking**: To access a jig inside a rack, you must `swap` edge jigs to other available racks.
* **Workflow**:
    1. **Unload**: Move `loaded` jigs from Beluga to racks.
    2. **Production**: Move `loaded` jigs from racks to craning hangars.
    3. **Return**: Send `empty` jigs from hangars back to the rack system.
    4. **Reload**: Load `empty` jigs from racks into the Beluga.

# PDDL State Schema
The `State` object contains two main collections for inspection:
* **Atoms (Predicates)**: Check existence via `(predicate, *args) in state.atoms`.
    * `('at', jig, location)`: Current position.
    * `('empty', jig)` / `('loaded', jig)`: Cargo status.
    * `('clear', jig, side)`: True if the jig is at a reachable rack edge.
    * `('on-trailer', jig, trailer)`: Jig is on a transport platform.
* **Fluents (Functions)**: Access via `state.fluents.get((fluent_name, *args))`.
    * `('occupied-slots', rack)`: Current count of jigs in a rack.

# Simplified API Reference
* `domain.get_applicable_actions(state)`: Returns a `Space` of valid actions.
* `domain.serialize_action(action)`: Returns a tuple, like `('move-jig-rack-to-trailer', 'jig1', 'rackA', 'trailer1')`.
* `domain.get_next_state(state, action)`: Returns the resulting deterministic `State`.
* `domain.get_transition_value(s, a, ns).cost`: Returns the positive cost of the action.
* `domain.is_goal(state)`: Returns True if the logistics mission is complete.


In [13]:
state = d.reset()

In [25]:
d.serialize_state(state)

TypeError: tuple indices must be integers or slices, not tuple

In [38]:
self = d

In [30]:
self.task.num_fluent_predicates

10

In [ ]:
def serialize_state(self, state: State) -> tuple[tuple[str], tuple[tuple[str], int]]:
    readable_atoms = tuple(
        (self.task.predicates[i].name, *(self.task.objects[o] for o in atom))
        for i, subatoms in enumerate(state.atoms)
        for atom in subatoms
    )
    readable_fluents = tuple(
        (
            (
                self.task.functions[i].name,
                *(self.task.objects[o] for o in f),
                value,
            )
            for i, subfluents in enumerate(state.fluents)
            for f, value in subfluents
        )
    )
    return readable_atoms, readable_fluents

In [41]:
state = d.reset()
len(state.fluents)

4

In [43]:
[f.name for f in self.task.functions]

['empty-size', 'free-space', 'size', 'total-cost']

In [45]:
state.fluents[0]

(((19,), 18),
 ((20,), 9),
 ((21,), 18),
 ((22,), 32),
 ((23,), 4),
 ((24,), 18),
 ((25,), 9),
 ((26,), 9),
 ((27,), 9),
 ((28,), 18),
 ((29,), 18),
 ((30,), 4),
 ((31,), 18),
 ((32,), 4),
 ((33,), 8),
 ((34,), 8),
 ((35,), 4),
 ((36,), 4),
 ((37,), 4),
 ((38,), 4),
 ((39,), 4),
 ((40,), 4),
 ((41,), 4),
 ((42,), 9),
 ((43,), 18),
 ((44,), 8),
 ((45,), 8),
 ((46,), 8),
 ((47,), 8),
 ((48,), 9),
 ((49,), 8),
 ((50,), 18),
 ((51,), 18),
 ((52,), 8),
 ((53,), 8),
 ((54,), 8),
 ((55,), 8),
 ((56,), 8),
 ((57,), 8),
 ((58,), 18),
 ((59,), 9),
 ((60,), 9),
 ((61,), 8),
 ((62,), 8),
 ((63,), 8),
 ((64,), 8),
 ((65,), 8),
 ((66,), 8),
 ((67,), 9),
 ((68,), 9),
 ((69,), 9),
 ((70,), 9),
 ((71,), 9),
 ((72,), 18),
 ((73,), 8),
 ((74,), 8),
 ((75,), 8),
 ((76,), 18),
 ((77,), 8),
 ((78,), 8),
 ((79,), 8),
 ((80,), 8),
 ((81,), 8),
 ((82,), 8),
 ((83,), 8),
 ((84,), 8),
 ((85,), 9),
 ((86,), 32),
 ((87,), 8),
 ((88,), 8),
 ((89,), 8),
 ((90,), 4),
 ((91,), 4),
 ((92,), 4),
 ((93,), 4),
 ((94,), 4)

In [47]:
readable_atoms = tuple(
    (self.task.predicates[i].name, *(self.task.objects[o] for o in atom))
    for i, subatoms in enumerate(state.atoms)
    for atom in subatoms
)
readable_fluents = tuple(
    (
        (
            self.task.functions[i].name,
            *(self.task.objects[o] for o in f),
            value,
        )
        for i, subfluents in enumerate(state.fluents)
        for f, value in subfluents
    )
)
readable_fluents

(('empty-size', 'jig0001', 18),
 ('empty-size', 'jig0002', 9),
 ('empty-size', 'jig0003', 18),
 ('empty-size', 'jig0004', 32),
 ('empty-size', 'jig0005', 4),
 ('empty-size', 'jig0006', 18),
 ('empty-size', 'jig0007', 9),
 ('empty-size', 'jig0008', 9),
 ('empty-size', 'jig0009', 9),
 ('empty-size', 'jig0010', 18),
 ('empty-size', 'jig0011', 18),
 ('empty-size', 'jig0012', 4),
 ('empty-size', 'jig0013', 18),
 ('empty-size', 'jig0014', 4),
 ('empty-size', 'jig0015', 8),
 ('empty-size', 'jig0016', 8),
 ('empty-size', 'jig0017', 4),
 ('empty-size', 'jig0018', 4),
 ('empty-size', 'jig0019', 4),
 ('empty-size', 'jig0020', 4),
 ('empty-size', 'jig0021', 4),
 ('empty-size', 'jig0022', 4),
 ('empty-size', 'jig0023', 4),
 ('empty-size', 'jig0024', 9),
 ('empty-size', 'jig0025', 18),
 ('empty-size', 'jig0026', 8),
 ('empty-size', 'jig0027', 8),
 ('empty-size', 'jig0028', 8),
 ('empty-size', 'jig0029', 8),
 ('empty-size', 'jig0030', 9),
 ('empty-size', 'jig0031', 8),
 ('empty-size', 'jig0032', 18),

In [50]:
readable_atoms

(('clear', 'jig0001', 'fside'),
 ('clear', 'jig0003', 'fside'),
 ('clear', 'jig0004', 'bside'),
 ('clear', 'jig0004', 'fside'),
 ('clear', 'jig0005', 'bside'),
 ('clear', 'jig0006', 'bside'),
 ('clear', 'jig0006', 'fside'),
 ('clear', 'jig0007', 'bside'),
 ('clear', 'jig0008', 'bside'),
 ('clear', 'jig0009', 'fside'),
 ('clear', 'jig0011', 'bside'),
 ('clear', 'jig0011', 'fside'),
 ('clear', 'jig0012', 'fside'),
 ('clear', 'jig0013', 'bside'),
 ('clear', 'jig0014', 'bside'),
 ('clear', 'jig0015', 'fside'),
 ('clear', 'jig0016', 'bside'),
 ('clear', 'jig0016', 'fside'),
 ('empty', 'beluga_trailer_1'),
 ('empty', 'beluga_trailer_2'),
 ('empty', 'beluga_trailer_3'),
 ('empty', 'factory_trailer_1'),
 ('empty', 'factory_trailer_2'),
 ('empty', 'jig0001'),
 ('empty', 'jig0002'),
 ('empty', 'jig0005'),
 ('empty', 'jig0007'),
 ('empty', 'jig0008'),
 ('empty', 'jig0011'),
 ('empty', 'jig0013'),
 ('empty', 'jig0014'),
 ('empty', 'hangar1'),
 ('empty', 'hangar2'),
 ('empty', 'hangar3'),
 ('in', '

In [29]:
len(s.atoms)

10

In [18]:
actions = d.get_applicable_actions()

In [20]:
d.serialize_action(actions.get_elements()[0])

(<plado.semantics.task.Action at 0x7fd0a1651810>,
 'jig0017',
 'jig0018',
 'beluga_trailer_2',
 'beluga1')

In [51]:
readable_a = d.serialize_action(a)
readable_a[0].name

'unload-beluga'

In [23]:
s = state
a = actions.get_elements()[0]
d.get_next_state(s, a)

In [7]:
from IPython.display import Markdown, display

display(
    Markdown(
        "You are an expert mathematician specialized in planning and scheduling.\nYour task is to improve a function that suggests the next action to take given the current position in a pddl domain.\nIt will be applied to a real-world logistic planning problem dealing with the storage and management of cargo transported by Airbus Beluga aircrafts.\n\nAircraft parts are held on *jigs*, which can slide and be stored on the *racks*.\n*Trailers* are used as special moving racks to transfer the jigs between the Beluga\nand the fixed racks. When the aircraft parts are sent to the production lines,\nthey transit to hangars where cranes remove the parts from the jigs.\nThe parts are then sent to production and the jigs return empty to the rack system.\n\nThe racks can contain several jigs in sequence, but only the jigs which are at the edges of the racks\n(either factory side or Beluga side) can be pulled out from the racks. This might require swapping jigs\nlocated at the rack edges to other racks in order to free the pass to jigs which are strictly inside the racks\n(i.e. not at their edges).\n\nConcerning the empty jigs to be returned to the Beluga, we only need to know their types\n(a jig type being a class of jigs of same loaded size and same empty size) and how many such\nempty jig types must be returned.\n\nWhen the Beluga lands on the production site, 2 high-level tasks must be performed:\n- unloading the parts (held on jigs) from the Beluga and storing them in the rack system ;\n- unstoring empty jigs from the rack system and loading them into the Beluga.\n\nBetween 2 Beluga flights, 3 high-level tasks must be considered:\n- unstoring parts held on jigs from the rack system and sending them to the production lines via the craning hangars ;\n- sending back empty jigs from the craning hangars to the rack system;\n- [optionally] swapping the jigs which are at the edges of the racks (either factory side or Beluga side) from one rack to another, in order to free the pass to jigs which are strictly inside the racks\n\nYour sampling function will be evaluated by applying it on several instances of the domain and computing the resulting average cost.\nMore precisely, given a maze, a new action is sampled and applied to the problem until the goal is reached\nor until a max number of steps related to the domain size (i.e. the number of jigs) is overcome. We take the average of the total cost on each test problem.\n\nThe heuristic is wrapped into a planner that can be used to stores information about the current problem from previous calls\nto increase the computation speed.\n\nYou must only perform edits between the # EVOLVE-BLOCK-START and # EVOLVE-BLOCK-END comments.\n\nYou are not allowed to use private attributes or methods of the domain, only its public API.\n\nFor context, here is the public api of <class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'> and of other relevant classes:\n## API Reference: `<class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'>`\n\n### Domain Capabilities (High-Level)\n- **Agent**: SingleAgent -> it is single-agent (i.e hosting only one agent).\n- **Concurrency**: Sequential -> its events/actions are sequential (non-parallel).\n- **Dynamics**: DeterministicTransitions -> its dynamics is deterministic and provided as a white-box model.\n- **Events**: Actions -> it handles only actions (i.e. controllable events).\n- **Goals**: Goals -> it has formalized goals.\n- **Initialization**: DeterministicInitialized -> it has a deterministic initial state known as white-box.\n- **Memory**: Markovian -> only its last state must be stored to compute its dynamics (pure Markovian\n- **Observability**: FullyObservable -> it is fully observable.\n- **Value**: PositiveCosts -> it sends only positive costs (i.e. negative rewards).\n\n### Domain base types\n- **T_event**: <class 'skd_domains.skd_base_domain.Action'>\n- **T_info**: None\n- **T_observation**: <class 'skd_domains.skd_base_domain.State'>\n- **T_predicate**: <class 'bool'>\n- **T_state**: <class 'skd_domains.skd_base_domain.State'>\n- **T_value**: <class 'float'>\n\n### Action space per agent\n<class 'skd_domains.skd_base_domain.ActionSpace'>\n\n### Observation space per agent\n<class 'skd_domains.skd_base_domain.ObservationSpace'>\n\n### Description\nDeterministic planning scikit-decide domain, based on a encoding of the actions to model the logics of\nthe transition function. The methods of this class should not be used directly, but rather through the\npublic methods of the domain feature class from which this class derives:\nhttps://airbus.github.io/scikit-decide/reference/_skdecide.domains.htmldeterministicplanningdomain\n\nArgs:\n    SkdBaseDomain (_type_): Base domain\n    DeterministicPlanningDomain (_type_): Compound scikit-decide class importing deterministic planning domain features\n\n### Attributes\n- **domain_str** (typing.Any): \n- **cost_functions** (set[int]): \n- **domain_path** (typing.Any): \n- **temp_pddl_directory** (<class 'NoneType'>): \n- **action_space** (typing.Any): \n- **transition_cost** (dict[tuple[State, Action, State], int]): \n- **observation_space** (typing.Any): \n- **task** (Task): \n- **problem_path** (typing.Any): \n- **succ_gen** (SuccessorGenerator): \n- **aops_gen** (ApplicableActionsGenerator): \n- **total_cost** (int | None): \n- **check_goal** (GoalChecker): \n\n### Methods\n#### - `__init__(self, beluga_problem: beluga_lib.beluga_problem.BelugaProblem, problem_name: str, instance_dir: os.PathLike = None, classic: bool = True, initial_state: beluga_lib.problem_state.BelugaProblemState = None) -> None`\n\n#### - `check_value(self, value: 'Value[D.T_value]') -> 'bool'`\nCheck that a value is compliant with its reward specification.\n\n**TIP:** This function returns always True by default because any kind of reward should be accepted at this level.\n\n##### Parameters\n- **value**: The value to check.\n\n##### Returns\nTrue if the value is compliant (False otherwise).\n\n#### - `cleanup(self)`\nErases the temporary directory containing PDDL files (if any)\n\n#### - `deserialize_action(self, action: tuple[str]) -> skd_domains.skd_base_domain.Action`\n\n#### - `deserialize_state(self, atoms: Iterable[tuple[str]], fluents: Iterable[tuple[tuple[str], int]])`\n\n#### - `get_action_mask(self, memory: 'Optional[D.T_state]' = None) -> 'Mask'`\nGet action mask for the given memory or internal one if omitted.\n\nAn action mask is another (more specific) format for applicable actions, that has a meaning only if the action\nspace can be iterated over in some way. It is represented by a flat array of 0's and 1's ordered as the actions\nwhen enumerated: 1 for an applicable action, and 0 for a not applicable action.\n\nThe action mask is used for instance by RL solvers to shut down logits associated to non-applicable actions in\nthe output of their internal neural network.\n\n##### Parameters\n- **memory**: The memory to consider. If None, works on the internal memory of the domain.\n\n##### Returns\na numpy array (or dict agent-> numpy array for multi-agent domains) with 0-1 indicating applicability of\nthe action (1 meaning applicable and 0 not applicable)\n\n#### - `get_action_space(self) -> 'Space[D.T_event]'`\nGet the (cached) domain action space (finite or infinite set).\n\n##### Returns\nThe action space.\n\n#### - `get_agents(self) -> 'set[str]'`\nReturn a singleton for single agent domains.\n\nWe must be here consistent with `skdecide.core.autocast()` which transforms a single agent domain\ninto a multi agents domain whose only agent has the id \"agent\".\n\n#### - `get_applicable_actions(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`\nGet the space (finite or infinite set) of applicable actions in the given memory (state or history), or in\nthe internal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nThe space of applicable actions.\n\n#### - `get_enabled_events(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`\nGet the space (finite or infinite set) of enabled uncontrollable events in the given memory (state or\nhistory), or in the internal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nThe space of enabled events.\n\n#### - `get_goals(self) -> 'Space[D.T_observation]'`\nGet the (cached) domain goals space (finite or infinite set).\n\n**WARNING:** Goal states are assumed to be fully observable (i.e. observation = state) so that there is never uncertainty\nabout whether the goal has been reached or not. This assumption guarantees that any policy that does not\nreach the goal with certainty incurs in infinite expected cost. - *Geffner, 2013: A Concise Introduction to\nModels and Methods for Automated Planning*\n\n##### Returns\nThe goals space.\n\n#### - `get_initial_state(self) -> 'D.T_state'`\nGet the (cached) initial state.\n\n##### Returns\nThe initial state.\n\n#### - `get_initial_state_distribution(self) -> 'Distribution[D.T_state]'`\nGet the (cached) probability distribution of initial states.\n\n##### Returns\nThe probability distribution of initial states.\n\n#### - `get_next_state(self, memory: 'D.T_state', action: 'D.T_event') -> 'D.T_state'`\nGet the next state given a memory and action.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe deterministic next state.\n\n#### - `get_next_state_distribution(self, memory: 'D.T_state', action: 'D.T_event') -> 'DiscreteDistribution[D.T_state]'`\nGet the discrete probability distribution of next state given a memory and action.\n\n**TIP:** In the Markovian case (memory only holds last state $s$), given an action $a$, this function can\nbe mathematically represented by $P(S'|s, a)$, where $S'$ is the next state random variable.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe discrete probability distribution of next state.\n\n#### - `get_observation(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'D.T_observation'`\nGet the deterministic observation given a state and action.\n\n##### Parameters\n- **state**: The state to be observed.\n- **action**: The last applied action (or None if the state is an initial state).\n\n##### Returns\nThe probability distribution of the observation.\n\n#### - `get_observation_distribution(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'Distribution[D.T_observation]'`\nGet the probability distribution of the observation given a state and action.\n\nIn mathematical terms (discrete case), given an action $a$, this function represents: $P(O|s, a)$,\nwhere $O$ is the random variable of the observation.\n\n##### Parameters\n- **state**: The state to be observed.\n- **action**: The last applied action (or None if the state is an initial state).\n\n##### Returns\nThe probability distribution of the observation.\n\n#### - `get_observation_space(self) -> 'Space[D.T_observation]'`\nGet the (cached) observation space (finite or infinite set).\n\n##### Returns\nThe observation space.\n\n#### - `get_pddl_domain(self) -> os.PathLike`\nGet the path to the PDDL domain file\n\nReturns:\n    os.PathLike: Path to the PDDL domain file\n\n#### - `get_pddl_problem(self) -> os.PathLike`\nGet the path to the PDDL problem file\n\nReturns:\n    os.PathLike: Path to the PDDL problem file\n\n#### - `get_transition_value(self, memory: 'D.T_state', action: 'D.T_event', next_state: 'Optional[D.T_state]' = None) -> 'Value[D.T_value]'`\nGet the value (reward or cost) of a transition.\n\nThe transition to consider is defined by the function parameters.\n\n**TIP:** If this function never depends on the next_state parameter for its computation, it is recommended to\nindicate it by overriding UncertainTransitions._is_transition_value_dependent_on_next_state_() to return\nFalse. This information can then be exploited by solvers to avoid computing next state to evaluate a\ntransition value (more efficient).\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n- **next_state**: The next state in which the transition ends (if needed for the computation).\n\n##### Returns\nThe transition value (reward or cost).\n\n#### - `is_action(self, event: 'D.T_event') -> 'bool'`\nIndicate whether an event is an action (i.e. a controllable event for the agents).\n\n**TIP:** \n\n##### Parameters\n- **event**: The event to consider.\n\n##### Returns\nTrue if the event is an action (False otherwise).\n\n#### - `is_applicable_action(self, action: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`\nIndicate whether an action is applicable in the given memory (state or history), or in the internal one if\nomitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nTrue if the action is applicable (False otherwise).\n\n#### - `is_enabled_event(self, event: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`\nIndicate whether an uncontrollable event is enabled in the given memory (state or history), or in the\ninternal one if omitted.\n\n##### Parameters\n- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).\n\n##### Returns\nTrue if the event is enabled (False otherwise).\n\n#### - `is_goal(self, observation: 'D.T_observation') -> 'D.T_predicate'`\nIndicate whether an observation belongs to the goals.\n\n**TIP:** \n\n##### Parameters\n- **observation**: The observation to consider.\n\n##### Returns\nTrue if the observation is a goal (False otherwise).\n\n#### - `is_observation(self, observation: 'D.T_observation') -> 'bool'`\nCheck that an observation indeed belongs to the domain observation space.\n\n**TIP:** \n\n##### Parameters\n- **observation**: The observation to consider.\n\n##### Returns\nTrue if the observation belongs to the domain observation space (False otherwise).\n\n#### - `is_terminal(self, state: 'D.T_state') -> 'D.T_predicate'`\nIndicate whether a state is terminal.\n\nA terminal state is a state with no outgoing transition (except to itself with value 0).\n\n##### Parameters\n- **state**: The state to consider.\n\n##### Returns\nTrue if the state is terminal (False otherwise).\n\n#### - `is_transition_value_dependent_on_next_state(self) -> 'bool'`\nIndicate whether get_transition_value() requires the next_state parameter for its computation (cached).\n\n##### Returns\nTrue if the transition value computation depends on next_state (False otherwise).\n\n#### - `reset(self) -> 'D.T_observation'`\nReset the state of the environment and return an initial observation.\n\n##### Returns\nAn initial observation.\n\n#### - `sample(self, memory: 'D.T_state', action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`\nSample one transition of the simulator's dynamics.\n\n##### Parameters\n- **memory**: The source memory (state or history) of the transition.\n- **action**: The action taken in the given memory (state or history) triggering the transition.\n\n##### Returns\nThe environment outcome of the sampled transition.\n\n#### - `serialize_action(self, action: skd_domains.skd_base_domain.Action) -> tuple[str]`\n\n#### - `serialize_state(self, state: skd_domains.skd_base_domain.State) -> tuple[tuple[str], tuple[tuple[str], int]]`\n\n#### - `set_memory(self, memory: 'D.T_state') -> 'None'`\nSet internal memory attribute _memory to given one.\n\nThis can be useful to set a specific \"starting point\" before doing a rollout with successive Environment.step()\ncalls.\n\n##### Parameters\n- **memory**: The memory to set internally.\n\n##### Example\n```python\n# Set simulation_domain memory to my_state (assuming Markovian domain)\nsimulation_domain.set_memory(my_state)\n\n# Start a 100-steps rollout from here (applying my_action at every step)\nfor _ in range(100):\n    simulation_domain.step(my_action)\n```\n\n#### - `step(self, action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`\nRun one step of the environment's dynamics.\n\n**WARNING:** Before calling Environment.step() the first time or when the end of an episode is\nreached, Initializable.reset() must be called to reset the environment's state.\n\n##### Parameters\n- **action**: The action taken in the current memory (state or history) triggering the transition.\n\n##### Returns\nThe environment outcome of this step.\n\n## API Reference: `<class 'skdecide.core.EnvironmentOutcome'>`\n\n### Description\nAn environment outcome for an internal transition.\n\n#### Parameters\n- **observation**: The agent's observation of the current environment.\n- **value**: The value (reward or cost) returned after previous action.\n- **termination**: Whether the episode has ended, in which case further step() calls will return undefined results.\n- **info**: Optional auxiliary diagnostic information (helpful for debugging, and sometimes learning).\n\n### Attributes\n- **info** (Optional[D.T_info]): \n- **observation** (D.T_observation): \n- **value** (Optional[D.T_value]): \n- **termination** (Optional[D.T_predicate]): \n\n### Methods\n#### - `__init__(self, observation: 'D.T_observation', value: 'Optional[D.T_value]' = None, termination: 'Optional[D.T_predicate]' = None, info: 'Optional[D.T_info]' = None) -> None`\n\n#### - `asdict(self)`\nReturn the fields of the instance as a new dictionary mapping field names to field values.\n\n#### - `astuple(self)`\nReturn the fields of the instance as a new tuple of field values.\n\n#### - `replace(self, **changes)`\nReturn a new object replacing specified fields with new values.\n\n## API Reference: `<class 'skdecide.core.DiscreteDistribution'>`\n\n### Description\nA discrete probability distribution.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, values: 'list[tuple[T, float]]') -> 'None'`\nInitialize DiscreteDistribution.\n\n**TIP:** If the given probabilities do not sum to 1, they are implicitly normalized as such for sampling.\n\n##### Parameters\n- **values**: The list of (element, probability) pairs.\n\n##### Example\n```python\ngame_strategy = DiscreteDistribution([('rock', 0.7), ('paper', 0.1), ('scissors', 0.2)])\nmove = game_strategy.sample()\n```\n\n#### - `get_values(self) -> 'list[tuple[T, float]]'`\nGet the list of (element, probability) pairs.\n\n##### Returns\nThe (element, probability) pairs.\n\n#### - `sample(self) -> 'T'`\n\n## API Reference: `<class 'skdecide.core.Value'>`\n\n### Description\nA value (reward or cost).\n\n**WARNING:** It is recommended to use either the reward or the cost parameter. If no one is used, a reward/cost of 0 is\nassumed. If both are used, reward will be considered and cost ignored. In any case, both reward and cost\nattributes will be defined after initialization.\n\n#### Parameters\n- **reward**: The optional reward.\n- **cost**: The optional cost.\n\n#### Example\n```python\n# These two lines are equivalent, use the one you prefer\nvalue_1 = Value(reward=-5)\nvalue_2 = Value(cost=5)\n\nassert value_1.reward == value_2.reward == -5  # True\nassert value_1.cost == value_2.cost == 5  # True\n```\n\n### Attributes\n- **reward** (Optional[D.T_value]): \n- **cost** (Optional[D.T_value]): \n\n### Methods\n#### - `__init__(self, reward: 'Optional[D.T_value]' = None, cost: 'Optional[D.T_value]' = None) -> None`\n\n## API Reference: `<class 'skdecide.core.Distribution'>`\n\n### Description\nA probability distribution.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, /, *args, **kwargs)`\nInitialize self.  See help(type(self)) for accurate signature.\n\n#### - `sample(self) -> 'T'`\nSample from this distribution.\n\n##### Returns\nThe sampled element.\n\n## API Reference: `<class 'skdecide.core.Space'>`\n\n### Description\nA space representing a finite or infinite set.\n\nThis class (or any of its descendant) is typically used to specify action, observation or goal spaces.\n\n### Attributes\nNone detected\n\n### Methods\n#### - `__init__(self, /, *args, **kwargs)`\nInitialize self.  See help(type(self)) for accurate signature.\n\n#### - `contains(self, x: 'T') -> 'bool'`\nCheck whether x is a valid member of this space.\n\n##### Parameters\n- **x**: The member to consider.\n\n##### Returns\nTrue if x is a valid member of this space (False otherwise)."
    )
)

You are an expert mathematician specialized in planning and scheduling.
Your task is to improve a function that suggests the next action to take given the current position in a pddl domain.
It will be applied to a real-world logistic planning problem dealing with the storage and management of cargo transported by Airbus Beluga aircrafts.

Aircraft parts are held on *jigs*, which can slide and be stored on the *racks*.
*Trailers* are used as special moving racks to transfer the jigs between the Beluga
and the fixed racks. When the aircraft parts are sent to the production lines,
they transit to hangars where cranes remove the parts from the jigs.
The parts are then sent to production and the jigs return empty to the rack system.

The racks can contain several jigs in sequence, but only the jigs which are at the edges of the racks
(either factory side or Beluga side) can be pulled out from the racks. This might require swapping jigs
located at the rack edges to other racks in order to free the pass to jigs which are strictly inside the racks
(i.e. not at their edges).

Concerning the empty jigs to be returned to the Beluga, we only need to know their types
(a jig type being a class of jigs of same loaded size and same empty size) and how many such
empty jig types must be returned.

When the Beluga lands on the production site, 2 high-level tasks must be performed:
- unloading the parts (held on jigs) from the Beluga and storing them in the rack system ;
- unstoring empty jigs from the rack system and loading them into the Beluga.

Between 2 Beluga flights, 3 high-level tasks must be considered:
- unstoring parts held on jigs from the rack system and sending them to the production lines via the craning hangars ;
- sending back empty jigs from the craning hangars to the rack system;
- [optionally] swapping the jigs which are at the edges of the racks (either factory side or Beluga side) from one rack to another, in order to free the pass to jigs which are strictly inside the racks

Your sampling function will be evaluated by applying it on several instances of the domain and computing the resulting average cost.
More precisely, given a maze, a new action is sampled and applied to the problem until the goal is reached
or until a max number of steps related to the domain size (i.e. the number of jigs) is overcome. We take the average of the total cost on each test problem.

The heuristic is wrapped into a planner that can be used to stores information about the current problem from previous calls
to increase the computation speed.

You must only perform edits between the # EVOLVE-BLOCK-START and # EVOLVE-BLOCK-END comments.

You are not allowed to use private attributes or methods of the domain, only its public API.

For context, here is the public api of <class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'> and of other relevant classes:
## API Reference: `<class 'skd_domains.skd_pddl_domain.SkdPDDLDomain'>`

### Domain Capabilities (High-Level)
- **Agent**: SingleAgent -> it is single-agent (i.e hosting only one agent).
- **Concurrency**: Sequential -> its events/actions are sequential (non-parallel).
- **Dynamics**: DeterministicTransitions -> its dynamics is deterministic and provided as a white-box model.
- **Events**: Actions -> it handles only actions (i.e. controllable events).
- **Goals**: Goals -> it has formalized goals.
- **Initialization**: DeterministicInitialized -> it has a deterministic initial state known as white-box.
- **Memory**: Markovian -> only its last state must be stored to compute its dynamics (pure Markovian
- **Observability**: FullyObservable -> it is fully observable.
- **Value**: PositiveCosts -> it sends only positive costs (i.e. negative rewards).

### Domain base types
- **T_event**: <class 'skd_domains.skd_base_domain.Action'>
- **T_info**: None
- **T_observation**: <class 'skd_domains.skd_base_domain.State'>
- **T_predicate**: <class 'bool'>
- **T_state**: <class 'skd_domains.skd_base_domain.State'>
- **T_value**: <class 'float'>

### Action space per agent
<class 'skd_domains.skd_base_domain.ActionSpace'>

### Observation space per agent
<class 'skd_domains.skd_base_domain.ObservationSpace'>

### Description
Deterministic planning scikit-decide domain, based on a encoding of the actions to model the logics of
the transition function. The methods of this class should not be used directly, but rather through the
public methods of the domain feature class from which this class derives:
https://airbus.github.io/scikit-decide/reference/_skdecide.domains.htmldeterministicplanningdomain

Args:
    SkdBaseDomain (_type_): Base domain
    DeterministicPlanningDomain (_type_): Compound scikit-decide class importing deterministic planning domain features

### Attributes
- **domain_str** (typing.Any): 
- **cost_functions** (set[int]): 
- **domain_path** (typing.Any): 
- **temp_pddl_directory** (<class 'NoneType'>): 
- **action_space** (typing.Any): 
- **transition_cost** (dict[tuple[State, Action, State], int]): 
- **observation_space** (typing.Any): 
- **task** (Task): 
- **problem_path** (typing.Any): 
- **succ_gen** (SuccessorGenerator): 
- **aops_gen** (ApplicableActionsGenerator): 
- **total_cost** (int | None): 
- **check_goal** (GoalChecker): 

### Methods
#### - `__init__(self, beluga_problem: beluga_lib.beluga_problem.BelugaProblem, problem_name: str, instance_dir: os.PathLike = None, classic: bool = True, initial_state: beluga_lib.problem_state.BelugaProblemState = None) -> None`

#### - `check_value(self, value: 'Value[D.T_value]') -> 'bool'`
Check that a value is compliant with its reward specification.

**TIP:** This function returns always True by default because any kind of reward should be accepted at this level.

##### Parameters
- **value**: The value to check.

##### Returns
True if the value is compliant (False otherwise).

#### - `cleanup(self)`
Erases the temporary directory containing PDDL files (if any)

#### - `deserialize_action(self, action: tuple[str]) -> skd_domains.skd_base_domain.Action`

#### - `deserialize_state(self, atoms: Iterable[tuple[str]], fluents: Iterable[tuple[tuple[str], int]])`

#### - `get_action_mask(self, memory: 'Optional[D.T_state]' = None) -> 'Mask'`
Get action mask for the given memory or internal one if omitted.

An action mask is another (more specific) format for applicable actions, that has a meaning only if the action
space can be iterated over in some way. It is represented by a flat array of 0's and 1's ordered as the actions
when enumerated: 1 for an applicable action, and 0 for a not applicable action.

The action mask is used for instance by RL solvers to shut down logits associated to non-applicable actions in
the output of their internal neural network.

##### Parameters
- **memory**: The memory to consider. If None, works on the internal memory of the domain.

##### Returns
a numpy array (or dict agent-> numpy array for multi-agent domains) with 0-1 indicating applicability of
the action (1 meaning applicable and 0 not applicable)

#### - `get_action_space(self) -> 'Space[D.T_event]'`
Get the (cached) domain action space (finite or infinite set).

##### Returns
The action space.

#### - `get_agents(self) -> 'set[str]'`
Return a singleton for single agent domains.

We must be here consistent with `skdecide.core.autocast()` which transforms a single agent domain
into a multi agents domain whose only agent has the id "agent".

#### - `get_applicable_actions(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`
Get the space (finite or infinite set) of applicable actions in the given memory (state or history), or in
the internal one if omitted.

##### Parameters
- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).

##### Returns
The space of applicable actions.

#### - `get_enabled_events(self, memory: 'Optional[D.T_state]' = None) -> 'Space[D.T_event]'`
Get the space (finite or infinite set) of enabled uncontrollable events in the given memory (state or
history), or in the internal one if omitted.

##### Parameters
- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).

##### Returns
The space of enabled events.

#### - `get_goals(self) -> 'Space[D.T_observation]'`
Get the (cached) domain goals space (finite or infinite set).

**WARNING:** Goal states are assumed to be fully observable (i.e. observation = state) so that there is never uncertainty
about whether the goal has been reached or not. This assumption guarantees that any policy that does not
reach the goal with certainty incurs in infinite expected cost. - *Geffner, 2013: A Concise Introduction to
Models and Methods for Automated Planning*

##### Returns
The goals space.

#### - `get_initial_state(self) -> 'D.T_state'`
Get the (cached) initial state.

##### Returns
The initial state.

#### - `get_initial_state_distribution(self) -> 'Distribution[D.T_state]'`
Get the (cached) probability distribution of initial states.

##### Returns
The probability distribution of initial states.

#### - `get_next_state(self, memory: 'D.T_state', action: 'D.T_event') -> 'D.T_state'`
Get the next state given a memory and action.

##### Parameters
- **memory**: The source memory (state or history) of the transition.
- **action**: The action taken in the given memory (state or history) triggering the transition.

##### Returns
The deterministic next state.

#### - `get_next_state_distribution(self, memory: 'D.T_state', action: 'D.T_event') -> 'DiscreteDistribution[D.T_state]'`
Get the discrete probability distribution of next state given a memory and action.

**TIP:** In the Markovian case (memory only holds last state $s$), given an action $a$, this function can
be mathematically represented by $P(S'|s, a)$, where $S'$ is the next state random variable.

##### Parameters
- **memory**: The source memory (state or history) of the transition.
- **action**: The action taken in the given memory (state or history) triggering the transition.

##### Returns
The discrete probability distribution of next state.

#### - `get_observation(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'D.T_observation'`
Get the deterministic observation given a state and action.

##### Parameters
- **state**: The state to be observed.
- **action**: The last applied action (or None if the state is an initial state).

##### Returns
The probability distribution of the observation.

#### - `get_observation_distribution(self, state: 'D.T_state', action: 'Optional[D.T_event]' = None) -> 'Distribution[D.T_observation]'`
Get the probability distribution of the observation given a state and action.

In mathematical terms (discrete case), given an action $a$, this function represents: $P(O|s, a)$,
where $O$ is the random variable of the observation.

##### Parameters
- **state**: The state to be observed.
- **action**: The last applied action (or None if the state is an initial state).

##### Returns
The probability distribution of the observation.

#### - `get_observation_space(self) -> 'Space[D.T_observation]'`
Get the (cached) observation space (finite or infinite set).

##### Returns
The observation space.

#### - `get_pddl_domain(self) -> os.PathLike`
Get the path to the PDDL domain file

Returns:
    os.PathLike: Path to the PDDL domain file

#### - `get_pddl_problem(self) -> os.PathLike`
Get the path to the PDDL problem file

Returns:
    os.PathLike: Path to the PDDL problem file

#### - `get_transition_value(self, memory: 'D.T_state', action: 'D.T_event', next_state: 'Optional[D.T_state]' = None) -> 'Value[D.T_value]'`
Get the value (reward or cost) of a transition.

The transition to consider is defined by the function parameters.

**TIP:** If this function never depends on the next_state parameter for its computation, it is recommended to
indicate it by overriding UncertainTransitions._is_transition_value_dependent_on_next_state_() to return
False. This information can then be exploited by solvers to avoid computing next state to evaluate a
transition value (more efficient).

##### Parameters
- **memory**: The source memory (state or history) of the transition.
- **action**: The action taken in the given memory (state or history) triggering the transition.
- **next_state**: The next state in which the transition ends (if needed for the computation).

##### Returns
The transition value (reward or cost).

#### - `is_action(self, event: 'D.T_event') -> 'bool'`
Indicate whether an event is an action (i.e. a controllable event for the agents).

**TIP:** 

##### Parameters
- **event**: The event to consider.

##### Returns
True if the event is an action (False otherwise).

#### - `is_applicable_action(self, action: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`
Indicate whether an action is applicable in the given memory (state or history), or in the internal one if
omitted.

##### Parameters
- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).

##### Returns
True if the action is applicable (False otherwise).

#### - `is_enabled_event(self, event: 'D.T_event', memory: 'Optional[D.T_state]' = None) -> 'bool'`
Indicate whether an uncontrollable event is enabled in the given memory (state or history), or in the
internal one if omitted.

##### Parameters
- **memory**: The memory to consider (if None, the internal memory attribute _memory is used instead).

##### Returns
True if the event is enabled (False otherwise).

#### - `is_goal(self, observation: 'D.T_observation') -> 'D.T_predicate'`
Indicate whether an observation belongs to the goals.

**TIP:** 

##### Parameters
- **observation**: The observation to consider.

##### Returns
True if the observation is a goal (False otherwise).

#### - `is_observation(self, observation: 'D.T_observation') -> 'bool'`
Check that an observation indeed belongs to the domain observation space.

**TIP:** 

##### Parameters
- **observation**: The observation to consider.

##### Returns
True if the observation belongs to the domain observation space (False otherwise).

#### - `is_terminal(self, state: 'D.T_state') -> 'D.T_predicate'`
Indicate whether a state is terminal.

A terminal state is a state with no outgoing transition (except to itself with value 0).

##### Parameters
- **state**: The state to consider.

##### Returns
True if the state is terminal (False otherwise).

#### - `is_transition_value_dependent_on_next_state(self) -> 'bool'`
Indicate whether get_transition_value() requires the next_state parameter for its computation (cached).

##### Returns
True if the transition value computation depends on next_state (False otherwise).

#### - `reset(self) -> 'D.T_observation'`
Reset the state of the environment and return an initial observation.

##### Returns
An initial observation.

#### - `sample(self, memory: 'D.T_state', action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`
Sample one transition of the simulator's dynamics.

##### Parameters
- **memory**: The source memory (state or history) of the transition.
- **action**: The action taken in the given memory (state or history) triggering the transition.

##### Returns
The environment outcome of the sampled transition.

#### - `serialize_action(self, action: skd_domains.skd_base_domain.Action) -> tuple[str]`

#### - `serialize_state(self, state: skd_domains.skd_base_domain.State) -> tuple[tuple[str], tuple[tuple[str], int]]`

#### - `set_memory(self, memory: 'D.T_state') -> 'None'`
Set internal memory attribute _memory to given one.

This can be useful to set a specific "starting point" before doing a rollout with successive Environment.step()
calls.

##### Parameters
- **memory**: The memory to set internally.

##### Example
```python
# Set simulation_domain memory to my_state (assuming Markovian domain)
simulation_domain.set_memory(my_state)

# Start a 100-steps rollout from here (applying my_action at every step)
for _ in range(100):
    simulation_domain.step(my_action)
```

#### - `step(self, action: 'D.T_event') -> 'EnvironmentOutcome[D.T_observation, Value[D.T_value], D.T_predicate, D.T_info]'`
Run one step of the environment's dynamics.

**WARNING:** Before calling Environment.step() the first time or when the end of an episode is
reached, Initializable.reset() must be called to reset the environment's state.

##### Parameters
- **action**: The action taken in the current memory (state or history) triggering the transition.

##### Returns
The environment outcome of this step.

## API Reference: `<class 'skdecide.core.EnvironmentOutcome'>`

### Description
An environment outcome for an internal transition.

#### Parameters
- **observation**: The agent's observation of the current environment.
- **value**: The value (reward or cost) returned after previous action.
- **termination**: Whether the episode has ended, in which case further step() calls will return undefined results.
- **info**: Optional auxiliary diagnostic information (helpful for debugging, and sometimes learning).

### Attributes
- **info** (Optional[D.T_info]): 
- **observation** (D.T_observation): 
- **value** (Optional[D.T_value]): 
- **termination** (Optional[D.T_predicate]): 

### Methods
#### - `__init__(self, observation: 'D.T_observation', value: 'Optional[D.T_value]' = None, termination: 'Optional[D.T_predicate]' = None, info: 'Optional[D.T_info]' = None) -> None`

#### - `asdict(self)`
Return the fields of the instance as a new dictionary mapping field names to field values.

#### - `astuple(self)`
Return the fields of the instance as a new tuple of field values.

#### - `replace(self, **changes)`
Return a new object replacing specified fields with new values.

## API Reference: `<class 'skdecide.core.DiscreteDistribution'>`

### Description
A discrete probability distribution.

### Attributes
None detected

### Methods
#### - `__init__(self, values: 'list[tuple[T, float]]') -> 'None'`
Initialize DiscreteDistribution.

**TIP:** If the given probabilities do not sum to 1, they are implicitly normalized as such for sampling.

##### Parameters
- **values**: The list of (element, probability) pairs.

##### Example
```python
game_strategy = DiscreteDistribution([('rock', 0.7), ('paper', 0.1), ('scissors', 0.2)])
move = game_strategy.sample()
```

#### - `get_values(self) -> 'list[tuple[T, float]]'`
Get the list of (element, probability) pairs.

##### Returns
The (element, probability) pairs.

#### - `sample(self) -> 'T'`

## API Reference: `<class 'skdecide.core.Value'>`

### Description
A value (reward or cost).

**WARNING:** It is recommended to use either the reward or the cost parameter. If no one is used, a reward/cost of 0 is
assumed. If both are used, reward will be considered and cost ignored. In any case, both reward and cost
attributes will be defined after initialization.

#### Parameters
- **reward**: The optional reward.
- **cost**: The optional cost.

#### Example
```python
# These two lines are equivalent, use the one you prefer
value_1 = Value(reward=-5)
value_2 = Value(cost=5)

assert value_1.reward == value_2.reward == -5  # True
assert value_1.cost == value_2.cost == 5  # True
```

### Attributes
- **reward** (Optional[D.T_value]): 
- **cost** (Optional[D.T_value]): 

### Methods
#### - `__init__(self, reward: 'Optional[D.T_value]' = None, cost: 'Optional[D.T_value]' = None) -> None`

## API Reference: `<class 'skdecide.core.Distribution'>`

### Description
A probability distribution.

### Attributes
None detected

### Methods
#### - `__init__(self, /, *args, **kwargs)`
Initialize self.  See help(type(self)) for accurate signature.

#### - `sample(self) -> 'T'`
Sample from this distribution.

##### Returns
The sampled element.

## API Reference: `<class 'skdecide.core.Space'>`

### Description
A space representing a finite or infinite set.

This class (or any of its descendant) is typically used to specify action, observation or goal spaces.

### Attributes
None detected

### Methods
#### - `__init__(self, /, *args, **kwargs)`
Initialize self.  See help(type(self)) for accurate signature.

#### - `contains(self, x: 'T') -> 'bool'`
Check whether x is a valid member of this space.

##### Parameters
- **x**: The member to consider.

##### Returns
True if x is a valid member of this space (False otherwise).